# MCP

## Review

We sent the agent text, images and audio as content blocks.

## Goals

Every tool we have used so far was a Python function we wrote ourselves.

The [Model Context Protocol](https://docs.langchain.com/oss/python/langchain/mcp) (MCP) "is an open protocol that standardizes how applications provide tools and context to language models."

An MCP server exposes tools, resources and prompts.

Any application that speaks MCP can use them, so a tool only has to be written once.

We'll connect agents to two MCP servers:

* A local server that lives in this repo
* A published reference server that tells the time

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

We'll use [LangSmith](https://docs.langchain.com/langsmith/home) for [tracing](https://docs.langchain.com/langsmith/observability-concepts).

We'll log to the project set by `LANGSMITH_PROJECT` in the repo-root `.env`, which is `ai-engineering`.

The next cell only matters on Windows.

It switches the event loop so Jupyter can launch MCP servers as subprocesses. On Linux and macOS it does nothing.

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

`resources/2.1_mcp_server.py` is a small MCP server built with FastMCP.

It exposes one of each thing an MCP server can offer:

* a tool, `search_web`, which searches the web with Tavily
* a resource, the README of `langchain-mcp-adapters`
* a prompt, which we'll use as the agent's system prompt

`MultiServerMCPClient` from [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters) connects to one or more servers.

Each server gets a name and a transport.

The `stdio` transport launches the server as a subprocess and talks to it over standard input and output.

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

The client's methods are async, so we `await` them.

Jupyter lets us use `await` directly in a cell.

* `get_tools` turns the server's tools into LangChain tools
* `get_resources` reads the server's resources
* `get_prompt` fetches a named prompt

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

The MCP tools are ordinary LangChain tools now, so they go straight into `create_agent`.

We use the server's prompt as the system prompt.

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=tools,
    system_prompt=prompt
)

MCP tools only run asynchronously, so we call the agent with `ainvoke` instead of `invoke`.

Calling `invoke` would fail with `NotImplementedError: StructuredTool does not support sync invocation.`

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='125bfb14-52ea-41df-9802-8b538e4c49e2'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Tell me about the langchain-mcp-adapters library". They want info about that library. It\'s presumably a library related to LangChain. We can search the web for it. Use search_web.', 'tool_calls': [{'id': 'fc_dfdcd17d-ebb9-4335-8257-f37fba0a52b0', 'function': {'arguments': '{"query":"langchain-mcp-adapters library"}', 'name': 'search_web'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 266, 'total_tokens': 341, 'completion_time': 0.076897277, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.014346484, 'prompt_tokens_details': None, 'queue_time': 0.202310617, 'total_time': 0.091243761}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_

## A published MCP server

We don't have to write MCP servers ourselves.

[`mcp-server-time`](https://github.com/modelcontextprotocol/servers/tree/main/src/time) is one of the reference servers from the MCP project.

Its tools are `get_current_time` and `convert_time`.

It is installed as a Python package in this repo, so we launch it over `stdio` with `uv run`.

`--local-timezone` sets the time zone the server treats as local.

In [8]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

This agent only gets the time server's tools, so it can tell the time but not search the web.

In [9]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=tools,
)

In [10]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='de20924e-932e-4eae-a76f-9cc28f0de626'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks "What time is it?" We should use get_current_time function. No timezone provided, so we should use default \'America/New_York\'.', 'tool_calls': [{'id': 'fc_8b3c5de0-f61b-481f-9b58-30eb5e087ff6', 'function': {'arguments': '{"timezone":"America/New_York"}', 'name': 'get_current_time'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 289, 'total_tokens': 348, 'completion_time': 0.105754669, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.015689268, 'prompt_tokens_details': None, 'queue_time': 0.2116315, 'total_time': 0.121443937}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

The model has no clock of its own.

It called `get_current_time` instead of guessing, and answered from the tool's result.